In [1]:
import pandas as pd
import pulp
import os
import joblib

In [2]:
MODEL_DIR = 'trained_models'
predicted_prices_tomorrow = {}

# loop semua komoditas, muat modelnya, dan tebak harganya
df =  pd.read_csv('dataset_clean.csv')
all_commodities = df['variant_nama'].unique()

for commodity in all_commodities:
    safe_filename = commodity.replace(" ", "_").replace("/", "_")
    model_path = os.path.join(MODEL_DIR, f"xgb_{safe_filename}.joblib")
    
    if os.path.exists(model_path):
        model = joblib.load(model_path)
        
        # Ambil baris data terakhir dari komoditas ini untuk menebak harga besok
        df_model = df[df['variant_nama'] == commodity]
        features = ['day', 'day_of_week', 'is_weekend', 'price_lag_1', 'price_lag_2', 'price_lag_3', 'rolling_mean_7d']
        
        latest_features = df_model[features].iloc[-1:].copy() 
        tomorrow_price_pred = model.predict(latest_features)[0]
        
        # Masukkan hasil prediksi asli ke dalam dictionary
        predicted_prices_tomorrow[commodity] = float(tomorrow_price_pred)

print(f"SUCCESS: Berhasil memprediksi harga untuk {len(predicted_prices_tomorrow)} komoditas.")

SUCCESS: Berhasil memprediksi harga untuk 16 komoditas.


C:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\pickle.py:1753: UserWarning: [12:35:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\gbm\../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


In [3]:
df_nutrition = pd.read_csv('dataset_gizi.csv')
df_nutrition.set_index('KOMODITAS', inplace=True)

available_items = [item for item in predicted_prices_tomorrow.keys() if item in df_nutrition.index]

In [ ]:
# Membuat model untuk meminimalkan biaya (LpMinimize)
lp_problem = pulp.LpProblem("NutriCost_Diet_Optimization", pulp.LpMinimize)

# Mendefinisikan Variabel Keputusan "Berapa Kg bahan X yang harus dibeli?"
# lowBound=0 memastikan tidak membeli bahan dalam jumlah minus
x = pulp.LpVariable.dicts("qty_kg", available_items, lowBound=0, cat='Continuous')

pakai_minyak_curah   = pulp.LpVariable('Pakai_Minyak_Curah',   cat='Binary')
pakai_minyak_premium = pulp.LpVariable('Pakai_Minyak_Premium', cat='Binary')
pakai_minyakita      = pulp.LpVariable('Pakai_Minyakita',      cat='Binary')

pakai_beras_premium  = pulp.LpVariable('Pakai_Beras_Premium',  cat='Binary')
pakai_beras_medium   = pulp.LpVariable('Pakai_Beras_Medium',   cat='Binary')

In [5]:
# 1. FUNGSI OBJEKTIF: Total Biaya = Harga * Kuantitas
lp_problem += pulp.lpSum([predicted_prices_tomorrow[i] * x[i] for i in available_items]), "Total_Cost"

# 2. KENDALA GIZI 1: Total Energi Minimal 2000 Kkal
lp_problem += pulp.lpSum([df_nutrition.loc[i, 'Energi'] * x[i] for i in available_items]) >= 2000, "Min_Energy"

# 3. KENDALA GIZI 2: Total Protein Minimal 50 gram (Cegah Stunting)
lp_problem += pulp.lpSum([df_nutrition.loc[i, 'Protein'] * x[i] for i in available_items]) >= 50, "Min_Protein"

# 4. KENDALA LOGIKA MANUSIA (Batas maksimal makan per hari per anak)
for i in available_items:
    if 'Beras' in i:
        lp_problem += x[i] <= 0.250  # Maksimal 250g
        # NOTE: Lower bound beras ditangani di bawah secara conditional via Big-M
    
    elif 'Terigu' in i:
        lp_problem += x[i] <= 0.050  # Maksimal 50g saja
        
    elif 'Gula' in i:
        lp_problem += x[i] <= 0.020
    elif 'Minyak' in i:
        lp_problem += x[i] <= 0.030
    elif 'Cabai' in i or 'Bawang' in i:
        lp_problem += x[i] <= 0.010
        
# Memaksa harus ada Protein Hewani (Ayam / Sapi / Telur) minimal 50 gram (0.050 kg)
protein_hewani_items = [i for i in available_items if 'Ayam' in i or 'Sapi' in i or 'Telur' in i]
if protein_hewani_items:
    lp_problem += pulp.lpSum([x[i] for i in protein_hewani_items]) >= 0.050

# ── CONSTRAINT MUTUAL EXCLUSION: Pilih Maks 1 Jenis per Grup ─────────────
lp_problem += pakai_minyak_curah + pakai_minyak_premium + pakai_minyakita <= 1, 'Pilih_Maks_1_Minyak'
lp_problem += pakai_beras_premium + pakai_beras_medium <= 1,                    'Pilih_Maks_1_Beras'

# ── CONSTRAINT BIG-M: Hubungkan biner ke variabel gramatur ───────────────
# Jika biner=0 -> qty dipaksa 0; jika biner=1 -> qty boleh s.d. M
M = 1.0

if 'Minyak Goreng Sawit Curah' in available_items:
    lp_problem += x['Minyak Goreng Sawit Curah']          <= M * pakai_minyak_curah,   'BigM_Minyak_Curah'
if 'Minyak Goreng Sawit Kemasan Premium' in available_items:
    lp_problem += x['Minyak Goreng Sawit Kemasan Premium'] <= M * pakai_minyak_premium, 'BigM_Minyak_Premium'
if 'Minyakita' in available_items:
    lp_problem += x['Minyakita']                           <= M * pakai_minyakita,      'BigM_Minyakita'

if 'Beras Premium' in available_items:
    lp_problem += x['Beras Premium'] <= M * pakai_beras_premium, 'BigM_Beras_Premium'
if 'Beras Medium' in available_items:
    lp_problem += x['Beras Medium']  <= M * pakai_beras_medium,  'BigM_Beras_Medium'

# ── FIX: CONDITIONAL LOWER BOUND untuk Beras (via Big-M) ─────────────────
# Jika biner=1 -> x >= 0.100 (minimal 100g beras); jika biner=0 -> x >= 0 (bebas)
# Ini menggantikan 'x[i] >= 0.100' yang unconditional (penyebab infeasible)
if 'Beras Premium' in available_items:
    lp_problem += x['Beras Premium'] >= 0.100 * pakai_beras_premium, 'MinBeras_Premium'
if 'Beras Medium' in available_items:
    lp_problem += x['Beras Medium']  >= 0.100 * pakai_beras_medium,  'MinBeras_Medium'

In [6]:
lp_problem.solve()

# Cek apakah mesin berhasil menemukan jalan keluar
if pulp.LpStatus[lp_problem.status] == 'Optimal':
    total_cost = pulp.value(lp_problem.objective)
    
    print(f"Total Biaya per Anak : Rp {total_cost:,.2f}")
    print("\nRekomendasi Resep & Gramatur")
    
    total_kcal = 0
    total_protein = 0
    
    for item in available_items:
        qty_kg = x[item].varValue
        if qty_kg > 0.001:  # Hanya tampilkan bahan yang dipakai lebih dari 1 gram
            qty_grams = qty_kg * 1000
            kcal_supplied = qty_kg * df_nutrition.loc[item, 'Energi']
            protein_supplied = qty_kg * df_nutrition.loc[item, 'Protein']
            cost_item = qty_kg * predicted_prices_tomorrow[item]
            
            print(f"- {item:.<30} {qty_grams:>5.0f} gram  (Rp {cost_item:>6,.0f})")
            
            total_kcal += kcal_supplied
            total_protein += protein_supplied
            
    print("\n[ Pencapaian Target Gizi ]")
    print(f"- Total Energi Terpenuhi   : {total_kcal:.1f} Kcal")
    print(f"- Total Protein Terpenuhi  : {total_protein:.1f} gram")
else:
    print("ERROR: Tidak dapat menemukan menu optimal")

Total Biaya per Anak : Rp 8,819.75

Rekomendasi Resep & Gramatur
- Gula Pasir Curah..............    20 gram  (Rp    360)
- Minyakita.....................    30 gram  (Rp    479)
- Telur Ayam Ras................    50 gram  (Rp  1,482)
- Tepung Terigu.................    50 gram  (Rp    622)
- Kedelai Impor.................   182 gram  (Rp  2,453)
- Beras Medium..................   250 gram  (Rp  3,424)

[ Pencapaian Target Gizi ]
- Total Energi Terpenuhi   : 2000.0 Kcal
- Total Protein Terpenuhi  : 86.6 gram
